# 03 — Causal tests + judge validation · One Dial or Many?

Notebook 02 found that the strict instruction's activation shift is **barely aligned** with the trained refusal direction (cos 0.16; it moves benign prompts only \~9% of the harmful-vs-harmless gap along it) — yet refusal starts jump from 7% to 67%. Geometry cannot say which part of the shift *causes* the refusals. This notebook does, on the **held-out eval split** (directions were fit on the fit split).

| Part | Question | Method | Output |
| --- | --- | --- | --- |
| **A** | Does the instruction still cause refusal when the trained direction is gone? (RQ4) | Fast refusal score, none / neutral / strict × normal / abliterated, on Hard-1K and toxic eval prompts | `scores/*.json`, fig 1 |
| **B** | Which part of the shift carries the refusals? | Add the strict − neutral shift (or only its along-direction / orthogonal part) to *neutral* prompts at the template tokens; remove them from *strict* prompts; sweep layers | fig 2 |
| **C** | Are the category "sub-dials" refusal mechanisms or just topic? | Add each category direction / residue to harmless prompts; compare with the main direction and random vectors | fig 3 |
| **D** | Does the fast score agree with real replies? | Generate \~440 replies for key conditions, label them blind with `judge.py`, compare | `gens/`, `labels/`, fig 4 |
| **E** | Hand-label sheet | 100 benign replies for you to label (judge accuracy) | `handlabel/` |

**Inputs:** the **IEdata** dataset (data files + `judge.py`) attached as a Kaggle input. **Secrets:** `HF_TOKEN`; `GLM_API_KEY` for Part D (if missing, Part D generates and saves replies and you can judge later, locally: `python judge.py <gens folder>`). **Runtime:** \~50 min on 2× T4. Every condition is saved as soon as it finishes; re-running skips finished work.

**Ethics:** abliterated replies to harmful prompts are generated only for aggregate labelling — they are never printed and are excluded from the hand-label sheet.

## 0 · Environment, storage, login

In [ ]:
import importlib, subprocess, sys
from importlib.metadata import version, PackageNotFoundError

def need(pkg, mm):
    try:
        return tuple(int(x) for x in version(pkg).split(".")[:2]) < mm
    except PackageNotFoundError:
        return True

to_install = [s for p, mm, s in [("transformers", (4, 50), "transformers>=4.50"),
                                 ("accelerate", (0, 26), "accelerate>=0.26")] if need(p, mm)]
if to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
    print("Installed:", to_install, "-> RESTART THE KERNEL, then run from the next cell.")
else:
    print({p: version(p) for p in ["transformers", "accelerate", "torch"]})

In [ ]:
import os, sys, json, time, math, random, glob, shutil
from contextlib import nullcontext, ExitStack
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_KAGGLE = os.path.exists("/kaggle")
IN_COLAB = (not IN_KAGGLE) and ("google.colab" in sys.modules or os.path.exists("/content"))
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/IE624"
elif IN_KAGGLE:
    ROOT = "/kaggle/working/IE624"
else:
    ROOT = os.path.abspath("IE624_runs")

DATA = f"{ROOT}/data"
RUN = f"{ROOT}/runs/03_causal_tests"
for d in [DATA] + [f"{RUN}/{s}" for s in ("stats", "scores", "gens", "labels", "reports", "handlabel")]:
    os.makedirs(d, exist_ok=True)
print("Platform:", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local", "| RUN:", RUN)

def get_secret(name):
    try:
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
    except Exception as e:
        print(f"secret {name} not available: {str(e)[:80]}")
    return os.environ.get(name)

token = get_secret("HF_TOKEN")
assert token, "Add HF_TOKEN to the notebook secrets."
from huggingface_hub import login
login(token=token)

In [ ]:
import numpy as np, pandas as pd, torch

def save_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, path)

def load_json(path, default=None):
    if not os.path.exists(path):
        return default
    with open(path) as f:
        return json.load(f)

def torch_save_atomic(obj, path):
    torch.save(obj, path + ".tmp"); os.replace(path + ".tmp", path)

def append_jsonl(path, rows):
    with open(path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, default=str, ensure_ascii=False) + "\n")
        f.flush(); os.fsync(f.fileno())

def read_jsonl(path):
    rows = []
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return rows

CHECKS_PATH = f"{RUN}/reports/checks.json"
CHECKS = load_json(CHECKS_PATH, default={})

def record(name, ok, **info):
    CHECKS[name] = {"ok": bool(ok), "time": time.strftime("%Y-%m-%d %H:%M:%S"), **info}
    save_json(CHECKS, CHECKS_PATH)
    print(("PASS " if ok else "FAIL ") + name, json.dumps(info, default=str)[:700])

RES_PATH = f"{RUN}/reports/results_03.json"
RES = load_json(RES_PATH, default={})

def save_res(**kv):
    RES.update(kv); save_json(RES, RES_PATH)

random.seed(0); np.random.seed(0); torch.manual_seed(0)

## 1 · Inputs (IEdata dataset, found wherever Kaggle mounted it)

In [ ]:
REQUIRED = ["orbench_toxic.parquet", "orbench_hard1k.parquet", "alpaca.parquet", "instructions.json",
            "refusal_direction.pt", "massive_dims.json"]
if any(not os.path.exists(f"{DATA}/{f}") for f in REQUIRED):
    hits = glob.glob("/kaggle/input/**/refusal_direction.pt", recursive=True)
    assert hits, "Attach the IEdata dataset: Add Input -> Your Datasets -> IEdata"
    src = os.path.dirname(hits[0])
    for f in os.listdir(src):
        if not os.path.exists(f"{DATA}/{f}") and os.path.isfile(os.path.join(src, f)):
            shutil.copy(os.path.join(src, f), DATA)
    print("copied inputs from", src)
missing = [f for f in REQUIRED if not os.path.exists(f"{DATA}/{f}")]
assert not missing, f"missing inputs: {missing}"

tox = pd.read_parquet(f"{DATA}/orbench_toxic.parquet")
hard = pd.read_parquet(f"{DATA}/orbench_hard1k.parquet")
alp = pd.read_parquet(f"{DATA}/alpaca.parquet")
INSTRUCTIONS = load_json(f"{DATA}/instructions.json")
EXCLUDE = load_json(f"{DATA}/category_policy.json", {"exclude_from_category_analysis": ["harmful"]})["exclude_from_category_analysis"]
SEL = torch.load(f"{DATA}/refusal_direction.pt")
MASSIVE = load_json(f"{DATA}/massive_dims.json")["massive_dims"]
L_STAR = int(SEL["layer"])
SEL_R = SEL["r"].float()                       # masked difference-in-means at layer L_STAR, <end_of_turn>
R_HAT = SEL_R / SEL_R.norm()

tox_fit, tox_eval = [d.reset_index(drop=True) for d in (tox[tox.split == "fit"], tox[tox.split == "eval"])]
alp_fit, alp_eval = [d.reset_index(drop=True) for d in (alp[alp.split == "fit"], alp[alp.split == "eval"])]
hard_fit, hard_eval = [d.reset_index(drop=True) for d in (hard[hard.split == "fit"], hard[hard.split == "eval"])]
print(f"direction: layer {L_STAR}, variant {SEL['variant']}, |r| = {SEL_R.norm():.1f}; {len(MASSIVE)} masked dims")
print({"tox": (len(tox_fit), len(tox_eval)), "alpaca": (len(alp_fit), len(alp_eval)), "hard": (len(hard_fit), len(hard_eval))})

# judge.py: from the dataset, or next to this notebook
JUDGE_DIR = None
for cand in glob.glob("/kaggle/input/**/judge.py", recursive=True) + glob.glob("judge.py") + glob.glob(f"{DATA}/judge.py"):
    JUDGE_DIR = os.path.dirname(os.path.abspath(cand)); break
print("judge.py:", JUDGE_DIR or "NOT FOUND (Part D will only generate; judge later)")

## 2 · Model

In [ ]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
assert torch.cuda.is_available(), "No GPU attached."
mems = [torch.cuda.get_device_properties(i).total_memory / 2**30 for i in range(torch.cuda.device_count())]
native_bf16 = all(torch.cuda.get_device_capability(i)[0] >= 8 for i in range(len(mems)))
DTYPE = torch.bfloat16 if native_bf16 else torch.float32 if sum(mems) >= 24 else torch.bfloat16

MODEL_ID = "google/gemma-3-4b-it"
BATCH = 16
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
DTYPE_KW = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"
load_kw = {DTYPE_KW: DTYPE, "device_map": "auto", "low_cpu_mem_usage": True}
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kw)
except Exception as e:
    print("AutoModelForCausalLM failed:", e)
    from transformers import Gemma3ForConditionalGeneration
    model = Gemma3ForConditionalGeneration.from_pretrained(MODEL_ID, **load_kw)
model.eval(); torch.set_grad_enabled(False)
print("loaded as", next(model.parameters()).dtype)

In [ ]:
# Model-dependent helpers (re-run this cell alone after any NameError).
assert RUN.endswith("03_causal_tests"), f"RUN = {RUN}: re-run the §0 cells of THIS notebook."
LAYERS = max(((n_, m) for n_, m in model.named_modules() if isinstance(m, torch.nn.ModuleList)
              and n_.endswith("layers") and "vision" not in n_), key=lambda x: len(x[1]))[1]
N_LAYERS = len(LAYERS)
D_MODEL = getattr(model.config, "text_config", model.config).hidden_size
INPUT_DEVICE = model.get_input_embeddings().weight.device
EOT_ID = tok.convert_tokens_to_ids("<end_of_turn>")
EOS_IDS = [i for i in {tok.eos_token_id, EOT_ID} if i is not None]
P = 5  # template suffix: <end_of_turn> \n <start_of_turn> model \n

def fmt(question, instruction=None):
    content = f"{instruction}\n\n{question}" if instruction else question
    return tok.apply_chat_template([{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True)

def encode(texts):
    return tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(INPUT_DEVICE)

REFUSAL_IDS = sorted({tok(s, add_special_tokens=False)["input_ids"][0] for s in ["I", "Sorry"]})
_NONREF = None

def refusal_score(logits):
    global _NONREF
    lp = torch.log_softmax(logits.float(), -1)
    if _NONREF is None or _NONREF.numel() != lp.shape[-1]:
        _NONREF = torch.ones(lp.shape[-1], dtype=torch.bool); _NONREF[REFUSAL_IDS] = False
    return torch.logsumexp(lp[:, REFUSAL_IDS], -1) - torch.logsumexp(lp[:, _NONREF], -1)

def forward(enc, hidden=False):
    try:
        return model(**enc, output_hidden_states=hidden, logits_to_keep=1)
    except TypeError:
        return model(**enc, output_hidden_states=hidden)

def masked(v):
    v = v.clone(); v[..., MASSIVE] = 0
    return v

def unit(v):
    return v / v.norm(dim=-1, keepdim=True).clamp_min(1e-12)

record("model_load", True, n_layers=N_LAYERS, d_model=D_MODEL, dtype=str(next(model.parameters()).dtype))

## 3 · Interventions

- **Abliteration** — remove the selected direction from every write into the residual stream (embeddings + Gemma's post-sublayer norms), all layers and positions (as validated in 01: KL 0.018).
- **Suffix patch** — add a vector to the residual stream entering layer *l* at the 5 template tokens (`<end_of_turn>` … `model\n`) on the prompt pass. This is where the strict − neutral shift was measured, so adding it there asks: *is the shift at these tokens enough to cause the refusals?*
- **ActAdd** — add a vector at layer *l*, all positions (Arditi-style steering; used for the category tests, as in 01).

In [ ]:
class DirectionAblation:
    def __init__(self, r):
        self.r = (r / r.norm()).float(); self._c = {}; self.handles = []
        self.targets = [model.get_input_embeddings()]
        for layer in LAYERS:
            self.targets += [layer.post_attention_layernorm, layer.post_feedforward_layernorm]

    def _hook(self, module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        r = self._c.setdefault((x.device, x.dtype), self.r.to(device=x.device, dtype=x.dtype))
        x = x - (x @ r).unsqueeze(-1) * r
        return (x,) + tuple(output[1:]) if isinstance(output, tuple) else x

    def __enter__(self):
        self.handles = [t.register_forward_hook(self._hook) for t in self.targets]; return self

    def __exit__(self, *exc):
        for h in self.handles:
            h.remove()


class _PreHook:
    def __init__(self, layer):
        self.module, self.handle, self._c = LAYERS[layer], None, {}

    def _vec(self, h):
        return self._c.setdefault((h.device, h.dtype), self.v.to(device=h.device, dtype=h.dtype))

    def _pre(self, module, args, kwargs):
        h = args[0] if args else kwargs["hidden_states"]
        h2 = self.apply(h)
        if h2 is None:
            return None
        if args:
            return (h2,) + tuple(args[1:]), kwargs
        kwargs["hidden_states"] = h2
        return args, kwargs

    def __enter__(self):
        self.handle = self.module.register_forward_pre_hook(self._pre, with_kwargs=True); return self

    def __exit__(self, *exc):
        self.handle.remove()


class ActAdd(_PreHook):
    def __init__(self, layer, v):
        super().__init__(layer); self.v = v.float()

    def apply(self, h):
        return h + self._vec(h)


class SuffixPatch(_PreHook):
    """Add vecs [P, d] to the last P positions on the prompt pass (decoding steps are left alone:
    the patched prompt states are already in the KV cache)."""
    def __init__(self, layer, vecs):
        super().__init__(layer); self.v = vecs.float()

    def apply(self, h):
        if h.shape[1] < self.v.shape[0]:
            return None
        h = h.clone(); h[:, -self.v.shape[0]:, :] += self._vec(h)
        return h


VECTORS = {}  # name -> (kind, layer, tensor)

def interventions(cond):
    stack = ExitStack()
    if cond.get("ablate"):
        stack.enter_context(DirectionAblation(SEL_R))
    if cond.get("vec"):
        kind, layer, t = VECTORS[cond["vec"]]
        stack.enter_context(SuffixPatch(layer, t) if kind == "suffix" else ActAdd(layer, t))
    return stack

def cond_name(cond):
    return f"{cond['set']}_{cond['instr']}_{'abl' if cond.get('ablate') else 'normal'}" + \
           (f"_{cond['vec']}" if cond.get("vec") else "")

SETS = {"hard": hard_eval, "tox": tox_eval, "alp": alp_eval}

@torch.no_grad()
def score(cond):
    """Fast refusal score for every prompt of cond['set'] under the condition. Cached per condition."""
    name = cond_name(cond); path = f"{RUN}/scores/{name}.json"
    cached = load_json(path)
    if cached is not None:
        return np.array(cached["scores"])
    df = SETS[cond["set"]]
    prompts = [fmt(q, INSTRUCTIONS[cond["instr"]]) for q in df["prompt"]]
    out = []
    for b in range(0, len(prompts), BATCH):
        enc = encode(prompts[b:b + BATCH])
        with interventions(cond):
            o = forward(enc)
        out.append(refusal_score(o.logits[:, -1, :].float().cpu()))
    s = torch.cat(out).numpy()
    save_json({"cond": cond, "ids": df["id"].tolist(), "scores": np.round(s, 4).tolist()}, path)
    return s

## 4 · Fit-split statistics: the strict − neutral shift and category directions

Residual stream at the 5 template tokens, for a sweep of layers, on the **fit** split (toxic + Alpaca under none; Hard-1K under neutral and strict). Cached in chunks.

In [ ]:
SWEEP_LAYERS = sorted({6, 10, 14, L_STAR, 20, 24})
LI = SWEEP_LAYERS.index(L_STAR)

@torch.no_grad()
def cache_stats(name, df, instr, chunk_size=128):
    out_dir = f"{RUN}/stats/{name}"; os.makedirs(out_dir, exist_ok=True)
    prompts = [fmt(q, INSTRUCTIONS[instr]) for q in df["prompt"]]
    for c in range(math.ceil(len(prompts) / chunk_size)):
        path = f"{out_dir}/chunk_{c:04d}.pt"
        if os.path.exists(path):
            continue
        s, e = c * chunk_size, min((c + 1) * chunk_size, len(prompts))
        H = []
        for b in range(s, e, BATCH):
            enc = encode(prompts[b:min(b + BATCH, e)])
            T = enc["input_ids"].shape[1]
            assert bool((enc["input_ids"][:, T - P] == EOT_ID).all())
            o = forward(enc, hidden=True)
            H.append(torch.stack([o.hidden_states[l][:, T - P:, :].float().cpu() for l in SWEEP_LAYERS], dim=1))
        torch_save_atomic(torch.cat(H), path)
        print(f"  {name}: chunk {c + 1}")
    return torch.cat([torch.load(f) for f in sorted(glob.glob(f"{out_dir}/chunk_*.pt"))])   # [n, |sweep|, P, d]

t0 = time.time()
ST = {"tox": cache_stats("tox_fit_none", tox_fit, "none"), "alp": cache_stats("alp_fit_none", alp_fit, "none"),
      "hard_neutral": cache_stats("hard_fit_neutral", hard_fit, "neutral"),
      "hard_strict": cache_stats("hard_fit_strict", hard_fit, "strict")}
print(f"stats ready in {time.time() - t0:.0f}s")

A_chk = masked(ST["tox"][:, LI, 0].mean(0) - ST["alp"][:, LI, 0].mean(0))
cos_chk = float(unit(A_chk) @ R_HAT)
record("direction_reproduced", cos_chk > 0.95, cos_with_selected=round(cos_chk, 4))

In [ ]:
# ---- the shift and its decomposition (layer L*, 5 template tokens) ----
S_raw = ST["hard_strict"].mean(0) - ST["hard_neutral"].mean(0)      # [|sweep|, P, d]
s = masked(S_raw[LI])                                               # [P, d]
coef = s @ R_HAT
s_par = coef[:, None] * R_HAT[None]
s_perp = s - s_par
g = torch.Generator().manual_seed(0)
rand = masked(torch.randn(P, D_MODEL, generator=g))
rand = unit(rand) * s_perp.norm(dim=-1, keepdim=True)

frac_par = (s_par.norm(dim=-1) ** 2 / s.norm(dim=-1) ** 2)
tokens = [repr(t) for t in ["<end_of_turn>", "\\n", "<start_of_turn>", "model", "\\n"]]
print("share of the shift's squared norm along the refusal direction, per template token:")
print(dict(zip(tokens, np.round(frac_par.numpy(), 4))))
print("shift norm per token:", np.round(s.norm(dim=-1).numpy(), 1), "| |r| =", round(float(SEL_R.norm()), 1))

VECTORS.update({
    "+s": ("suffix", L_STAR, s), "+s_par": ("suffix", L_STAR, s_par), "+s_perp": ("suffix", L_STAR, s_perp),
    "+rand": ("suffix", L_STAR, rand), "+s_unmasked": ("suffix", L_STAR, S_raw[LI]),
    "-s": ("suffix", L_STAR, -s), "-s_par": ("suffix", L_STAR, -s_par), "-s_perp": ("suffix", L_STAR, -s_perp),
})
for i, l in enumerate(SWEEP_LAYERS):
    VECTORS[f"+s@L{l}"] = ("suffix", l, masked(S_raw[i]))

# ---- category directions and residues (layer L*, <end_of_turn>) ----
cats_all = tox_fit["category"].values
CATS = sorted(c for c in set(cats_all) if c not in EXCLUDE)
mu_s = ST["alp"][:, LI, 0].mean(0)
r_norm = SEL_R.norm()
for c in CATS:
    C = masked(ST["tox"][torch.from_numpy(cats_all == c), LI, 0].mean(0) - mu_s)
    res = C - (C @ R_HAT) * R_HAT
    VECTORS[f"cat:{c}"] = ("actadd", L_STAR, C)
    VECTORS[f"res:{c}"] = ("actadd", L_STAR, unit(res) * r_norm)
VECTORS["A"] = ("actadd", L_STAR, SEL_R)
for k in range(3):
    VECTORS[f"randdir{k}"] = ("actadd", L_STAR, unit(masked(torch.randn(D_MODEL, generator=g))) * r_norm)

save_res(shift={"frac_sq_norm_along_r": frac_par.tolist(), "shift_norm": s.norm(dim=-1).tolist(),
                "r_norm": float(r_norm), "sweep_layers": SWEEP_LAYERS}, categories=CATS)

## Part A · Does the instruction still cause refusal without the trained direction?

Fast refusal score (share of replies likely to *start* as a refusal) on held-out prompts. The key quantity is the **instruction effect** = strict − neutral, with and without abliteration. If abliteration wipes out harmful refusal but the instruction effect survives, the instruction refuses through a different route.

In [ ]:
t0 = time.time()
GRID = [{"set": "hard", "instr": i, "ablate": a} for i in ("none", "neutral", "strict") for a in (False, True)] + \
       [{"set": "tox", "instr": i, "ablate": a} for i in ("none", "strict") for a in (False, True)]
SC = {}
for cnd in GRID:
    SC[cond_name(cnd)] = score(cnd)
    print(f"  {cond_name(cnd):28s} refusal-start rate {np.mean(SC[cond_name(cnd)] > 0):.3f}  ({time.time() - t0:.0f}s)")

def rate(name):
    return float(np.mean(SC[name] > 0))

def wilson(p, n, z=1.96):
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h

def boot_diff(a, b, B=2000, seed=0):
    """CI for mean(a > 0) - mean(b > 0), paired over prompts."""
    rng = np.random.default_rng(seed); n = len(a)
    ia, ib = (a > 0).astype(float), (b > 0).astype(float)
    idx = rng.integers(0, n, (B, n))
    d = ia[idx].mean(1) - ib[idx].mean(1)
    return float(np.quantile(d, 0.025)), float(np.quantile(d, 0.975))

eff_normal = rate("hard_strict_normal") - rate("hard_neutral_normal")
eff_abl = rate("hard_strict_abl") - rate("hard_neutral_abl")
ci_normal = boot_diff(SC["hard_strict_normal"], SC["hard_neutral_normal"])
ci_abl = boot_diff(SC["hard_strict_abl"], SC["hard_neutral_abl"])
tox_drop = rate("tox_none_normal") - rate("tox_none_abl")
partA = {"rates": {k: rate(k) for k in SC}, "instruction_effect_normal": eff_normal, "ci_normal": ci_normal,
         "instruction_effect_abliterated": eff_abl, "ci_abliterated": ci_abl,
         "survival_ratio": eff_abl / eff_normal if eff_normal else None, "toxic_refusal_drop": tox_drop}
save_res(partA=partA)
record("abliteration_removes_harmful_refusal", rate("tox_none_abl") < 0.5 * rate("tox_none_normal"),
       toxic_normal=rate("tox_none_normal"), toxic_abl=rate("tox_none_abl"))
record("instruction_effect_survives_abliteration", True, normal=round(eff_normal, 3), ci_normal=ci_normal,
       abliterated=round(eff_abl, 3), ci_abl=ci_abl, survival_ratio=partA["survival_ratio"])

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, (setname, instrs, title) in zip(ax, [("hard", ("none", "neutral", "strict"), "Hard-1K eval (benign)"),
                                            ("tox", ("none", "strict"), "OR-Bench toxic eval (harmful)")]):
    x = np.arange(len(instrs)); w = 0.38
    for j, (abl, col, lab) in enumerate([(False, "tab:blue", "normal"), (True, "tab:orange", "abliterated")]):
        names = [f"{setname}_{i}_{'abl' if abl else 'normal'}" for i in instrs]
        ys = [rate(nm) for nm in names]
        cis = [wilson(y, len(SC[nm])) for y, nm in zip(ys, names)]
        a.bar(x + (j - 0.5) * w, ys, w, color=col, label=lab,
              yerr=np.array([[y - lo for y, (lo, hi) in zip(ys, cis)], [hi - y for y, (lo, hi) in zip(ys, cis)]]), capsize=3)
        for xi, y in zip(x + (j - 0.5) * w, ys):
            a.text(xi, y + 0.02, f"{y:.0%}", ha="center", fontsize=8)
    a.set_xticks(x); a.set_xticklabels(instrs); a.set_ylim(0, 1.05); a.set_title(title); a.set_ylabel("refusal-start rate")
    a.legend()
plt.tight_layout(); fig.savefig(f"{RUN}/reports/fig1_abliteration_grid.png", dpi=130); plt.show()
print(f"instruction effect (strict − neutral): normal {eff_normal:+.3f} {ci_normal}, abliterated {eff_abl:+.3f} {ci_abl}")

## Part B · Which part of the shift carries the refusals?

**Recovery** = (rate(neutral + patch) − rate(neutral)) / (rate(strict) − rate(neutral)): the share of the strict instruction's refusal effect reproduced by adding a piece of the shift to *neutral* prompts at the template tokens. **Removal** = the share of the effect lost when a piece is subtracted from *strict* prompts. Pieces: the full masked shift, its component **along** the refusal direction, the **orthogonal** remainder, a random vector of the orthogonal part's size, and the unmasked shift. A layer sweep shows where the full shift is sufficient.

In [ ]:
t0 = time.time()
B_CONDS = [{"set": "hard", "instr": "neutral", "vec": v} for v in ("+s", "+s_par", "+s_perp", "+rand", "+s_unmasked")] + \
          [{"set": "hard", "instr": "strict", "vec": v} for v in ("-s", "-s_par", "-s_perp")] + \
          [{"set": "hard", "instr": "neutral", "vec": f"+s@L{l}"} for l in SWEEP_LAYERS]
for cnd in B_CONDS:
    SC[cond_name(cnd)] = score(cnd)
    print(f"  {cond_name(cnd):32s} {np.mean(SC[cond_name(cnd)] > 0):.3f}  ({time.time() - t0:.0f}s)")

base_n, base_s = SC["hard_neutral_normal"], SC["hard_strict_normal"]

def frac_ci(cond_scores, mode, B=2000, seed=0):
    rng = np.random.default_rng(seed); n = len(base_n)
    a, b, c = (base_n > 0).astype(float), (base_s > 0).astype(float), (cond_scores > 0).astype(float)
    def f(ia, ib, ic):
        gap = ib - ia
        return (ic - ia) / gap if mode == "recovery" else (ib - ic) / gap
    point = f(a.mean(), b.mean(), c.mean())
    idx = rng.integers(0, n, (B, n))
    bs = f(a[idx].mean(1), b[idx].mean(1), c[idx].mean(1))
    return float(point), float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))

partB = {}
for v in ("+s", "+s_par", "+s_perp", "+rand", "+s_unmasked"):
    partB[f"recovery {v}"] = frac_ci(SC[f"hard_neutral_normal_{v}"], "recovery")
for v in ("-s", "-s_par", "-s_perp"):
    partB[f"removal {v}"] = frac_ci(SC[f"hard_strict_normal_{v}"], "removal")
sweep = {l: frac_ci(SC[f"hard_neutral_normal_+s@L{l}"], "recovery") for l in SWEEP_LAYERS}
save_res(partB=partB, partB_layer_sweep={str(k): v for k, v in sweep.items()})
for k, (p, lo, hi) in partB.items():
    print(f"  {k:22s} {p:+.2f}  [{lo:+.2f}, {hi:+.2f}]")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
keys = list(partB)
vals = np.array([partB[k] for k in keys])
cols = ["tab:red" if "par" in k else "tab:purple" if "perp" in k else "grey" if "rand" in k else "tab:blue" for k in keys]
ax[0].barh(range(len(keys)), vals[:, 0], xerr=[vals[:, 0] - vals[:, 1], vals[:, 2] - vals[:, 0]], color=cols, capsize=3)
ax[0].set_yticks(range(len(keys))); ax[0].set_yticklabels(keys); ax[0].axvline(0, c="k", lw=0.5); ax[0].axvline(1, c="k", ls=":", lw=0.8)
ax[0].set_title(f"share of the strict instruction's refusal effect (layer {L_STAR}, template tokens)")
sv = np.array([sweep[l] for l in SWEEP_LAYERS])
ax[1].errorbar(SWEEP_LAYERS, sv[:, 0], yerr=[sv[:, 0] - sv[:, 1], sv[:, 2] - sv[:, 0]], marker="o", capsize=3)
ax[1].axhline(0, c="k", lw=0.5); ax[1].axhline(1, c="k", ls=":", lw=0.8)
ax[1].set_xlabel("layer"); ax[1].set_title("recovery from adding the full shift at one layer")
plt.tight_layout(); fig.savefig(f"{RUN}/reports/fig2_shift_decomposition.png", dpi=130); plt.show()
record("shift_decomposition", True, **{k: round(v[0], 3) for k, v in partB.items()})

## Part C · Are the category sub-dials refusal mechanisms, or just topic?

Add each vector at layer L* (all positions) to **harmless Alpaca eval prompts**: the main direction A, each full category direction, each category **residue** (A removed, scaled to |A|), and random vectors of the same size. If residues cause refusal well beyond random vectors, the sub-dials are refusal mechanisms; if they behave like random vectors, they encode topic.

In [ ]:
t0 = time.time()
C_VECS = ["A"] + [f"cat:{c}" for c in CATS] + [f"res:{c}" for c in CATS] + [f"randdir{k}" for k in range(3)]
SC["alp_none_normal"] = score({"set": "alp", "instr": "none"})
for v in C_VECS:
    SC[f"alp_none_normal_{v}"] = score({"set": "alp", "instr": "none", "vec": v})
print(f"done in {time.time() - t0:.0f}s")
partC = {v: {"rate": rate(f"alp_none_normal_{v}"), "mean_score": float(np.mean(SC[f"alp_none_normal_{v}"]))}
         for v in ["none"] + C_VECS if v != "none"}
partC["none"] = {"rate": rate("alp_none_normal"), "mean_score": float(np.mean(SC["alp_none_normal"]))}
save_res(partC=partC)

fig, a = plt.subplots(figsize=(13, 4))
order = ["none", "A"] + [f"cat:{c}" for c in CATS] + [f"res:{c}" for c in CATS] + [f"randdir{k}" for k in range(3)]
cols = ["grey", "black"] + ["tab:red"] * len(CATS) + ["tab:purple"] * len(CATS) + ["lightgrey"] * 3
a.bar(range(len(order)), [partC[k]["rate"] for k in order], color=cols)
a.set_xticks(range(len(order))); a.set_xticklabels(order, rotation=70, ha="right", fontsize=8)
a.set_ylabel("refusal-start rate on harmless prompts"); a.set_ylim(0, 1.05)
a.set_title(f"inducing refusal at layer {L_STAR}: red = full category direction, purple = residue (|A|-sized), light = random (|A|-sized)")
plt.tight_layout(); fig.savefig(f"{RUN}/reports/fig3_category_induce.png", dpi=130); plt.show()
res_rates = [partC[f"res:{c}"]["rate"] for c in CATS]; rnd_rates = [partC[f"randdir{k}"]["rate"] for k in range(3)]
record("category_residues_induce", True, residue_mean=round(float(np.mean(res_rates)), 3),
       residue_max=round(float(np.max(res_rates)), 3), random_mean=round(float(np.mean(rnd_rates)), 3),
       A=round(partC["A"]["rate"], 3), baseline=round(partC["none"]["rate"], 3))

## Part D · Real replies, judged blind

Generate replies (≤ 200 tokens, greedy) for the key conditions on a stratified subset of the eval prompts, then label them with `judge.py` (it sees only the question and the reply, never the condition). Replies are appended per batch; re-running skips finished ones.

In [ ]:
N_HARD, N_TOX = 60, 40
rng = np.random.default_rng(0)
hard_sub = (hard_eval.groupby("category", group_keys=False)
            .apply(lambda d: d.sample(min(len(d), max(2, round(N_HARD * len(d) / len(hard_eval)))), random_state=0)))
hard_sub = hard_sub.sample(min(N_HARD, len(hard_sub)), random_state=0).reset_index(drop=True)
tox_sub = tox_eval.sample(N_TOX, random_state=0).reset_index(drop=True)
SUBS = {"hard": hard_sub, "tox": tox_sub}

GEN_CONDS = [{"set": "hard", "instr": "neutral"}, {"set": "hard", "instr": "strict"},
             {"set": "hard", "instr": "neutral", "ablate": True}, {"set": "hard", "instr": "strict", "ablate": True},
             {"set": "hard", "instr": "neutral", "vec": "+s_par"}, {"set": "hard", "instr": "neutral", "vec": "+s_perp"},
             {"set": "tox", "instr": "none"}, {"set": "tox", "instr": "none", "ablate": True}]

@torch.no_grad()
def generate(cond, batch_size=8, max_new_tokens=200):
    name = cond_name(cond); path = f"{RUN}/gens/{name}.jsonl"
    df = SUBS[cond["set"]]
    done = {r["id"] for r in read_jsonl(path)}
    todo = df[~df["id"].isin(done)]
    for b in range(0, len(todo), batch_size):
        chunk = todo.iloc[b:b + batch_size]
        enc = encode([fmt(q, INSTRUCTIONS[cond["instr"]]) for q in chunk["prompt"]])
        with interventions(cond):
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 eos_token_id=EOS_IDS, pad_token_id=tok.pad_token_id)
        new = out[:, enc["input_ids"].shape[1]:].cpu()
        append_jsonl(path, [{"id": i, "prompt": q, "category": c, "condition": name, "set": cond["set"],
                             "instruction": cond["instr"], "ablate": bool(cond.get("ablate")), "vec": cond.get("vec"),
                             "response": tok.decode(seq, skip_special_tokens=True).strip()}
                            for i, q, c, seq in zip(chunk["id"], chunk["prompt"], chunk["category"], new)])
    return read_jsonl(path)

t0 = time.time()
GENS = {}
for cnd in GEN_CONDS:
    GENS[cond_name(cnd)] = generate(cnd)
    print(f"  {cond_name(cnd):32s} {len(GENS[cond_name(cnd)])} replies  ({(time.time() - t0) / 60:.1f} min)")

# benign examples only (never print abliterated replies to harmful prompts)
ex = hard_sub.iloc[0]
for nm in [cond_name(c) for c in GEN_CONDS if c["set"] == "hard"]:
    r = next(x for x in GENS[nm] if x["id"] == ex["id"])
    print(f"[{nm}] {r['response'][:200]!r}")

In [ ]:
# ---- judge (needs judge.py + GLM_API_KEY); safe to skip and run locally later ----
LABELS_PATH = f"{RUN}/labels/labels.jsonl"
all_rows = [r for v in GENS.values() for r in v]
key = get_secret("GLM_API_KEY")
if JUDGE_DIR and key:
    os.environ["GLM_API_KEY"] = key
    sys.path.insert(0, JUDGE_DIR)
    import judge
    print("judge:", judge.MODEL, "at", judge.BASE)
    judge.judge_rows(all_rows, LABELS_PATH, workers=8)
else:
    import hashlib
    print("Skipping the judge (judge.py or GLM_API_KEY missing). Download runs/03_causal_tests/gens and run:\n"
          "  python judge.py <gens folder> --out labels.jsonl\nthen put labels.jsonl into runs/03_causal_tests/labels/.")

## Fast score vs judge

For every judged reply: does the fast score (reply likely starts with "I"/"Sorry") agree with the judge's verdict (redirect / lecture / hard_refuse = refusal)? Reported per condition, plus overall AUROC of the fast score for predicting a judged refusal.

In [ ]:
def row_key_local(prompt, response):
    if "judge" in globals():
        return judge.row_key(prompt, response)
    import hashlib
    model_name = os.environ.get("GLM_MODEL", "hyde/glm-5.3-flash")
    return hashlib.sha1(f"v1\x00{model_name}\x00{prompt}\x00{response}".encode("utf-8")).hexdigest()

labels = {r["key"]: r for r in read_jsonl(LABELS_PATH) if r.get("label") not in (None, "error", "parse_error")}
if not labels:
    print("No labels yet - run the judge cell (or judge locally) first.")
else:
    rows = []
    for nm, gens in GENS.items():
        sc = load_json(f"{RUN}/scores/{nm}.json")
        smap = dict(zip(sc["ids"], sc["scores"])) if sc else {}
        for r in gens:
            lab = labels.get(row_key_local(r["prompt"], r["response"]))
            if lab is None:
                continue
            rows.append({"condition": nm, "set": r["set"], "id": r["id"], "label": lab["label"],
                         "judged_refusal": lab["label"] in ("redirect", "lecture", "hard_refuse"),
                         "fast_score": smap.get(r["id"], np.nan)})
    J = pd.DataFrame(rows)
    J["fast_refusal"] = J["fast_score"] > 0

    def auroc(s, y):
        s, y = np.asarray(s, float), np.asarray(y, bool)
        if y.all() or (~y).all():
            return float("nan")
        rk = pd.Series(s).rank().values
        return float((rk[y].sum() - y.sum() * (y.sum() + 1) / 2) / (y.sum() * (~y).sum()))

    summary = J.groupby("condition").agg(n=("id", "size"), fast_rate=("fast_refusal", "mean"),
                                         judged_rate=("judged_refusal", "mean"),
                                         agreement=("fast_refusal", lambda s: float((s == J.loc[s.index, "judged_refusal"]).mean())))
    styles = pd.crosstab(J["condition"], J["label"], normalize="index").round(2)
    overall = {"n": len(J), "agreement": float((J.fast_refusal == J.judged_refusal).mean()),
               "auroc": auroc(J.fast_score, J.judged_refusal),
               "fast_precision": float(J[J.fast_refusal].judged_refusal.mean()) if J.fast_refusal.any() else None,
               "fast_recall": float(J[J.judged_refusal].fast_refusal.mean()) if J.judged_refusal.any() else None}
    print(summary.round(3).to_string()); print(); print(styles.to_string()); print(); print(overall)
    save_res(judge_validation={"overall": overall, "by_condition": summary.round(4).reset_index().to_dict("records"),
                               "styles": styles.reset_index().to_dict("records")})
    record("fast_score_vs_judge", overall["auroc"] > 0.8 if overall["auroc"] == overall["auroc"] else False, **overall)

    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    sm = summary.loc[[cond_name(c) for c in GEN_CONDS if cond_name(c) in summary.index]]
    x = np.arange(len(sm)); w = 0.38
    ax[0].bar(x - w / 2, sm.fast_rate, w, label="fast score"); ax[0].bar(x + w / 2, sm.judged_rate, w, label="judge")
    ax[0].set_xticks(x); ax[0].set_xticklabels(sm.index, rotation=40, ha="right", fontsize=8); ax[0].legend()
    ax[0].set_title("refusal rate: fast score vs judge"); ax[0].set_ylim(0, 1.05)
    st = styles.loc[sm.index]
    bottom = np.zeros(len(st))
    for lab, col in zip(["comply", "comply_warning", "redirect", "lecture", "hard_refuse", "other"],
                        ["tab:green", "yellowgreen", "gold", "orange", "tab:red", "grey"]):
        if lab in st:
            ax[1].bar(x, st[lab], bottom=bottom, color=col, label=lab); bottom += st[lab].values
    ax[1].set_xticks(x); ax[1].set_xticklabels(st.index, rotation=40, ha="right", fontsize=8)
    ax[1].legend(fontsize=7, bbox_to_anchor=(1, 1)); ax[1].set_title("judged response styles")
    plt.tight_layout(); fig.savefig(f"{RUN}/reports/fig4_judge_validation.png", dpi=130); plt.show()

## Part E · Hand-label sheet (100 benign replies)

Exports `handlabel/handlabel_sheet.csv` (question + reply + an empty `your_label` column) and a separate answer key. Label without looking at the key; then the agreement between you and the judge goes in the report. Replies to harmful prompts are excluded.

In [ ]:
SHEET = f"{RUN}/handlabel/handlabel_sheet.csv"
if os.path.exists(SHEET):
    print("sheet already exists:", SHEET)
else:
    pool = [r for nm, g in GENS.items() for r in g if r["set"] == "hard"]
    pick = random.Random(0).sample(pool, min(100, len(pool)))
    sheet = pd.DataFrame([{"item": i, "question": r["prompt"], "reply": r["response"], "your_label": ""}
                          for i, r in enumerate(pick)])
    keyf = pd.DataFrame([{"item": i, "condition": r["condition"], "key": row_key_local(r["prompt"], r["response"])}
                         for i, r in enumerate(pick)])
    sheet.to_csv(SHEET, index=False)
    keyf.to_csv(f"{RUN}/handlabel/handlabel_key_DO_NOT_OPEN.csv", index=False)
    print("wrote", SHEET, "- labels: comply, comply_warning, redirect, lecture, hard_refuse, other")

## Summary

In [ ]:
print("=== Part A: instruction effect (strict − neutral, refusal-start rate on benign Hard-1K eval) ===")
print(f"  normal model : {RES['partA']['instruction_effect_normal']:+.3f}  CI {RES['partA']['ci_normal']}")
print(f"  abliterated  : {RES['partA']['instruction_effect_abliterated']:+.3f}  CI {RES['partA']['ci_abliterated']}")
print(f"  survival ratio {RES['partA']['survival_ratio']}; toxic refusal drop under abliteration {RES['partA']['toxic_refusal_drop']:.3f}")
print("=== Part B: share of the instruction's effect (point, 95% CI) ===")
for k, v in RES["partB"].items():
    print(f"  {k:22s} {v[0]:+.2f} [{v[1]:+.2f}, {v[2]:+.2f}]")
print("=== Part C: harmless prompts, refusal-start rate when adding ===")
print({k: round(v["rate"], 3) for k, v in RES["partC"].items()})
if "judge_validation" in RES:
    print("=== Part D: fast score vs judge ===", RES["judge_validation"]["overall"])
for k, v in CHECKS.items():
    print(f"{k:42s} {'PASS' if v['ok'] else 'FAIL'}")
print("\nSaved under:", RUN)

### Reading the results

| Pattern | Meaning |
| --- | --- |
| Toxic refusal collapses under abliteration **and** the strict − neutral effect stays large | The instruction refuses through a **separate route** — it survives removal of the trained dial |
| The instruction effect also collapses | The instruction **works through the trained dial**, despite the low cosine (a small push along it is enough) |
| `recovery +s_par` ≫ `recovery +s_perp` | The small along-direction component carries the refusals |
| `recovery +s_perp` ≫ `recovery +s_par` | The orthogonal 97% carries them — **another dial** |
| `recovery +s` small at every layer | The instruction's effect is not localised at the template tokens of one layer (it acts through earlier tokens / several layers) |
| Residues induce refusal ≫ random vectors | Category sub-dials are **refusal** mechanisms, not just topic |
| AUROC of the fast score ≥ 0.8 vs the judge | The fast-score numbers in 01–03 can be trusted (report the judge's rates as the final ones) |